# Thesis — Stage 1 Weighted + Stage 2 Oversampling + Threshold, with Optuna

This notebook adds **Optuna hyperparameter tuning** on top of the base experiment.
The EDA, preprocessing, train/validation/test splits, tokenization, and metrics are
**identical** to the base notebook. The only additions are:

- an Optuna `study` per stage that searches over `learning_rate`, `num_train_epochs`,
  `per_device_train_batch_size`, and `weight_decay`;
- each trial trains on the **train** split and is scored on the **validation** split
  (never the test set), so tuning never sees the held-out test data;
- Stage 1 optimizes toxic-class **F1**; Stage 2 optimizes **macro F1**;
- after the search, the best hyperparameters are used to train a final model, which is
  then evaluated once on the untouched **test** set (with decision-threshold tuning
  done on validation).

`N_TRIALS` is kept small so the notebook finishes in reasonable time on Kaggle; raise it
for a more thorough search.

**This variant keeps all 5 Stage-2 labels.** Stage 1 uses a class-weighted
CrossEntropyLoss; Stage 2 uses random oversampling of the training split. Both stages
add per-label / toxic-class decision-threshold tuning on validation.

In [4]:
# Install Optuna if it isn't already available in the environment.
# On Kaggle this is usually a no-op after the first run.
!pip install -q optuna

In [5]:
import pandas as pd
import numpy as np
import tqdm

from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
from huggingface_hub import login

import torch
from torch import nn
import torch.nn.functional as F
from sklearn.utils.class_weight import compute_class_weight

import optuna

import ast
import re

# Silence transformers' informational/warning logs (e.g. legacy LayerNorm
# gamma/beta key-name notices when Trainer reloads the best checkpoint).
# Doesn't affect training/evaluation, only console noise.
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

In [6]:
df = pd.read_csv("/kaggle/input/datasets/evelynuntariady/thesis/data_thesis_majority_vote.csv")

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26379 entries, 0 to 26378
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   Unnamed: 0                     26379 non-null  int64 
 1   text_id                        26379 non-null  object
 2   annotators_id                  26379 non-null  object
 3   text                           26379 non-null  object
 4   topic                          26379 non-null  object
 5   is_noise_or_spam_text          26379 non-null  int64 
 6   toxicity                       26379 non-null  int64 
 7   polarized                      26379 non-null  int64 
 8   profanity_obscenity            26379 non-null  int64 
 9   threat_incitement_to_violence  26379 non-null  int64 
 10  insults                        26379 non-null  int64 
 11  identity_attack                26379 non-null  int64 
 12  sexually_explicit              26379 non-null  int64 
 13  c

In [8]:
df.head()

,Unnamed: 0,text_id,annotators_id,text,topic,is_noise_or_spam_text,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit,cleaned_text,processed_text
0,0,2026-01-01 00:00:00,"['7', '15']",Wajah Jurnalis Dibalut Perban Saat Melaporkan ...,UNKNOWN,0,0,0,0,0,0,0,0,Wajah jurnalis dibalut perban saat melaporkan ...,wajah jurnalis dibalut perban saat melaporkan ...
1,1,2026-01-10 00:00:00,"['7', '15']",Elektabilitas Paslon 02 sebentar lagi terjun b...,Disabilitas,0,0,1,0,0,0,0,0,Elektabilitas pasangan calon 02 sebentar lagi ...,elektabilitas pasangan calon sebentar lagi ter...
2,2,1-1001,"['10', '18']","Kristen, Hindu, Islam dapat perlakuan istimewa...",Kristen,0,0,1,0,0,0,0,0,"Kristen, Hindu, Islam dapat perlakuan istimewa...","kristen, hindu, islam dapat perlakuan istimewa..."
3,3,1-1004,"['10', '18']","Masyarakat Kristen, Hindu, Budha, dan etnis Ti...","Kristen, Tionghoa",0,0,0,0,0,0,0,0,"Masyarakat Kristen, Hindu, Buddha, dan etnis T...","masyarakat kristen, hindu, buddha, dan etnis t..."
4,4,1-1005,"['10', '18']",Cina Mengirimkan Satelit Penginderaan Jauh Bar...,Tionghoa,0,0,0,0,0,0,0,0,Cina mengirimkan satelit penginderaan jauh bar...,cina mengirimkan satelit penginderaan jauh bar...


In [9]:
def transform_datetime_id(text_id_str):
    # Regex to match the datetime pattern like 'YYYY-MM-DD HH:MM:SS'
    match = re.match(r'(\d+)-(\d{2})-(\d{2}) \d{2}:\d{2}:\d{2}', str(text_id_str))
    if match:
        year_part = match.group(1)
        month_part = int(match.group(2))
        day_part = int(match.group(3))

        if year_part == '2026':
            # For year 2026, format as month-day (e.g. '1-1')
            return f"{month_part}-{day_part}"
        else:
            # For other years, format as month-original_year (e.g. '2-8783')
            return f"{month_part}-{year_part}"
    else:
        return text_id_str  # Return original if pattern doesn't match

# Apply this transformation to the 'text_id' column of your DataFrame
df['text_id'] = df['text_id'].apply(transform_datetime_id)

In [10]:
cols = [
'is_noise_or_spam_text',
'toxicity',
'polarized',
'profanity_obscenity',
'threat_incitement_to_violence',
'insults',
'identity_attack',
'sexually_explicit'
]

for col in cols:
    df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x,str) else x)

In [11]:
df_main = df.copy()

In [12]:
df_main = df_main[df_main['is_noise_or_spam_text'] == 0]

In [13]:
df_main.isnull().sum()

Unnamed: 0                       0
text_id                          0
annotators_id                    0
text                             0
topic                            0
is_noise_or_spam_text            0
toxicity                         0
polarized                        0
profanity_obscenity              0
threat_incitement_to_violence    0
insults                          0
identity_attack                  0
sexually_explicit                0
cleaned_text                     0
processed_text                   0
dtype: int64

In [14]:
slang_dict = {
    "gk": "tidak",
    "ga": "tidak",
    "nggak": "tidak",
    "yg": "yang",
    "lu": "kamu",
    "loe": "kamu",
    "gw": "saya",
    "gue": "saya",
    "bgt": "banget",
    "dgn": "dengan",
    "krn": "karena",
    "aja": "saja",
    "trs": "terus"
}

In [15]:
import re

def normalize_slang(text, slang_dict):
    words = text.split()
    return " ".join([slang_dict.get(w.lower(), w) for w in words])

def preprocess_text(text, slang_dict, lowercase=True):
    if not isinstance(text, str):
        return ""

    # 1. remove username (@user)
    text = re.sub(r'@\w+', '', text)

    # 2. remove URL
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # 3. remove numbers
    text = re.sub(r'\d+', '', text)

    # 4. remove unwanted symbols (keep .,!?)
    text = re.sub(r'[^\w\s.,!?]', '', text)

    # 5. normalize slang
    text = normalize_slang(text, slang_dict)

    # 6. normalize repeated chars (contoh: anjiiing -> anjiing)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 7. lowercase (optional)
    if lowercase:
        text = text.lower()

    # 8. remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [16]:
from tqdm import tqdm
tqdm.pandas()

df_main["processed_text"] = (
    df_main["cleaned_text"]
    .fillna("")
    .astype(str)
    .progress_apply(
        lambda x: preprocess_text(x, slang_dict)
    )
)

100%|██████████| 26379/26379 [00:01<00:00, 21654.15it/s]


In [17]:
df_main['toxicity'].value_counts()

toxicity
0    23980
1     2399
Name: count, dtype: int64

In [18]:
df_main['profanity_obscenity'].value_counts()

profanity_obscenity
0    26018
1      361
Name: count, dtype: int64

In [19]:
df_main['insults'].value_counts()

insults
0    25315
1     1064
Name: count, dtype: int64

In [20]:
df_main['identity_attack'].value_counts()

identity_attack
0    25365
1     1014
Name: count, dtype: int64

# Stage 1 — Binary Toxicity (Weighted + Optuna)

In [ ]:
from huggingface_hub import login

login("")

In [ ]:
x = df_main['processed_text']
y = df_main['toxicity']

In [ ]:
x_train_full, x_test, y_train_full, y_test = train_test_split(
    x, y, test_size=0.2, stratify=y, random_state=42
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=0.125, stratify=y_train_full, random_state=42
)

print("Stage 1 split sizes:", {"train": len(x_train), "val": len(x_val), "test": len(x_test)})

In [ ]:
# Class weights for Stage 1, computed from the TRAIN split only, used by the weighted
# CrossEntropyLoss below.
class_weights_s1 = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights_s1 = torch.tensor(class_weights_s1, dtype=torch.float)
print("Stage 1 class weights (class 0, class 1):", class_weights_s1)

In [ ]:
train_df = pd.DataFrame({"processed_text": x_train, "label": y_train})
val_df = pd.DataFrame({"processed_text": x_val, "label": y_val})
test_df = pd.DataFrame({"processed_text": x_test, "label": y_test})

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

In [ ]:
from transformers import (
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
)
tokenizer = AutoTokenizer.from_pretrained("indolem/indobertweet-base-uncased")

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["processed_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [ ]:
tokenized_datasets_1 = dataset.map(tokenize, batched=True)

tokenized_datasets_1 = tokenized_datasets_1.remove_columns(["processed_text"])
tokenized_datasets_1 = tokenized_datasets_1.rename_column("label", "labels")
tokenized_datasets_1.set_format("torch")

train_dataset_1 = tokenized_datasets_1["train"]
val_dataset_1 = tokenized_datasets_1["validation"]
test_dataset_1 = tokenized_datasets_1["test"]

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, auc, roc_curve, precision_recall_curve
)

def compute_metrics_s1(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
        "recall": recall_score(labels, preds),
        "precision": precision_score(labels, preds),
        "roc_auc": auc(roc_curve(labels, preds)[0], roc_curve(labels, preds)[1])
    }

In [ ]:
# Trainer that applies class weights to CrossEntropyLoss, used to handle label
# imbalance in Stage 1.
class WeightedTrainerSingleLabel(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = nn.CrossEntropyLoss(weight=weight)
        # logits.size(-1) instead of model.config.num_labels: stays correct even if
        # Trainer wraps the model in nn.DataParallel (which doesn't expose .config).
        loss = loss_fct(logits.view(-1, logits.size(-1)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

## Optuna search — Stage 1 (weighted loss)

In [ ]:
N_TRIALS = 10  # raise for a more thorough search

def model_init_s1():
    return AutoModelForSequenceClassification.from_pretrained(
        "indolem/indobertweet-base-uncased",
        num_labels=2
    )

def objective_s1(trial):
    import shutil, os
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
    epochs = trial.suggest_int("num_train_epochs", 3, 6)
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [8, 16])
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)

    args = TrainingArguments(
        output_dir=f"./optuna_s1/trial_{trial.number}",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=16,
        learning_rate=lr,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        save_only_model=True,      # don't persist optimizer state during search (saves disk)
        save_total_limit=1,        # keep only the best checkpoint per trial
        metric_for_best_model="f1",
        greater_is_better=True,
        remove_unused_columns=False,
        logging_steps=50,
        disable_tqdm=True,
        report_to=[],
    )

    trainer = WeightedTrainerSingleLabel(
        model_init=model_init_s1,
        args=args,
        train_dataset=train_dataset_1,
        eval_dataset=val_dataset_1,
        compute_metrics=compute_metrics_s1,
        class_weights=class_weights_s1,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    try:
        trainer.train()
        metrics = trainer.evaluate()
        return metrics["eval_f1"]
    finally:
        # Free disk between trials: Kaggle's working dir fills up fast, and a
        # half-written checkpoint is what triggers the torch.save iostream error.
        shutil.rmtree(f"./optuna_s1/trial_{trial.number}", ignore_errors=True)

study_s1 = optuna.create_study(direction="maximize", study_name="stage1_weighted")
study_s1.optimize(objective_s1, n_trials=N_TRIALS)

print("Best Stage 1 params:", study_s1.best_params)
print("Best Stage 1 validation F1:", study_s1.best_value)

## Train final Stage 1 model with best hyperparameters

In [ ]:
best_s1 = study_s1.best_params

args_s1 = TrainingArguments(
    output_dir="./best_s1",
    num_train_epochs=best_s1["num_train_epochs"],
    per_device_train_batch_size=best_s1["per_device_train_batch_size"],
    per_device_eval_batch_size=16,
    learning_rate=best_s1["learning_rate"],
    weight_decay=best_s1["weight_decay"],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    remove_unused_columns=False,
    logging_steps=50,
    disable_tqdm=False,
    report_to=[],
)

model_s1 = model_init_s1()
trainer_s1 = WeightedTrainerSingleLabel(
    model=model_s1,
    args=args_s1,
    train_dataset=train_dataset_1,
    eval_dataset=val_dataset_1,
    compute_metrics=compute_metrics_s1,
    class_weights=class_weights_s1,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer_s1.train()

In [ ]:
# Threshold tuning on VALIDATION for Stage 1.
pred1_val = trainer_s1.predict(val_dataset_1)
probs1_val = torch.softmax(torch.tensor(pred1_val.predictions), dim=-1)[:, 1].numpy()

precision_curve_s1, recall_curve_s1, thresholds_s1 = precision_recall_curve(y_val, probs1_val)
f1_curve_s1 = 2 * precision_curve_s1 * recall_curve_s1 / (precision_curve_s1 + recall_curve_s1 + 1e-12)
best_idx_s1 = np.argmax(f1_curve_s1[:-1])
best_threshold_s1 = thresholds_s1[best_idx_s1]

print(f"Stage 1 best threshold (max F1 on validation): {best_threshold_s1:.4f}")
print(f"  validation precision={precision_curve_s1[best_idx_s1]:.4f} "
      f"recall={recall_curve_s1[best_idx_s1]:.4f} f1={f1_curve_s1[best_idx_s1]:.4f}")

In [ ]:
pred1 = trainer_s1.predict(test_dataset_1)
pred_label1 = pred1.predictions.argmax(axis=1)

probs1_test = torch.softmax(torch.tensor(pred1.predictions), dim=-1)[:, 1].numpy()
pred_label1_tuned = (probs1_test >= best_threshold_s1).astype(int)

print("Stage 1 classification report @ default threshold (0.5) -- test set:")
print(classification_report(y_test, pred_label1))

print(f"\nStage 1 classification report @ tuned threshold ({best_threshold_s1:.4f}) -- test set:")
print(classification_report(y_test, pred_label1_tuned))

In [ ]:
trainer_s1.save_model("./model_stage1_weighted_optuna")
tokenizer.save_pretrained("./model_stage1_weighted_optuna")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.create_repo(
    repo_id="evelnap/hs-stage1-weighted-optuna",
    repo_type="model",
    exist_ok=True
)

api.upload_folder(
    folder_path="./model_stage1_weighted_optuna",
    repo_id="evelnap/hs-stage1-weighted-optuna",
    repo_type="model"
)

# Stage 2 — 5-Label Toxicity Type (Oversampling + Optuna + Threshold)

In [22]:
df_toxic = df_main[df_main["toxicity"] == 1].copy()

In [23]:
# This variant keeps ALL 5 labels.
LABEL_COLS = [
    "insults",
    "identity_attack",
    "threat_incitement_to_violence",
    "profanity_obscenity",
    "sexually_explicit",
]

X = df_toxic["processed_text"]
y2 = df_toxic[LABEL_COLS]

In [24]:
train_val_df_s2, test_df_s2 = train_test_split(df_toxic, test_size=0.2, random_state=42)
train_df_s2, val_df_s2 = train_test_split(train_val_df_s2, test_size=0.125, random_state=42)

train_df_s2 = train_df_s2[["processed_text"] + LABEL_COLS].copy()
val_df_s2 = val_df_s2[["processed_text"] + LABEL_COLS].copy()
test_df_s2 = test_df_s2[["processed_text"] + LABEL_COLS].copy()

print("Stage 2 split sizes:", {"train": len(train_df_s2), "val": len(val_df_s2), "test": len(test_df_s2)})

Stage 2 split sizes: {'train': 1679, 'val': 240, 'test': 480}


In [25]:
# Random oversampling of minority-label rows for Stage 2 (multi-label). For each label
# whose positive count is below the most frequent label's count, duplicate (with
# replacement) rows carrying that label until the gap is filled. Only the TRAIN split is
# resampled -- val/test keep their natural distribution.
def oversample_multilabel(df, label_cols, random_state=42):
    max_count = df[label_cols].sum().max()
    parts = [df]
    for col in label_cols:
        col_count = df[col].sum()
        n_needed = int(max_count - col_count)
        if n_needed > 0:
            minority_rows = df[df[col] == 1]
            if len(minority_rows) > 0:
                parts.append(minority_rows.sample(n=n_needed, replace=True, random_state=random_state))
    balanced = pd.concat(parts).sample(frac=1, random_state=random_state).reset_index(drop=True)
    return balanced

In [26]:
# Oversample the TRAIN split only. val/test keep their natural distribution.
print("Before oversampling:", {c: int(train_df_s2[c].sum()) for c in LABEL_COLS})
train_df_s2 = oversample_multilabel(train_df_s2, LABEL_COLS, random_state=42)
print("After oversampling:", {c: int(train_df_s2[c].sum()) for c in LABEL_COLS})

Before oversampling: {'insults': 729, 'identity_attack': 698, 'threat_incitement_to_violence': 78, 'profanity_obscenity': 258, 'sexually_explicit': 47}
After oversampling: {'insults': 1028, 'identity_attack': 1161, 'threat_incitement_to_violence': 740, 'profanity_obscenity': 977, 'sexually_explicit': 744}


In [27]:
train_df_s2["label"] = train_df_s2[LABEL_COLS].astype(float).values.tolist()
val_df_s2["label"] = val_df_s2[LABEL_COLS].astype(float).values.tolist()
test_df_s2["label"] = test_df_s2[LABEL_COLS].astype(float).values.tolist()

train_df_s2 = train_df_s2[["processed_text", "label"]]
val_df_s2 = val_df_s2[["processed_text", "label"]]
test_df_s2 = test_df_s2[["processed_text", "label"]]

In [28]:
dataset_s2 = DatasetDict({
    "train": Dataset.from_pandas(train_df_s2.reset_index(drop=True)),
    "validation": Dataset.from_pandas(val_df_s2.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df_s2.reset_index(drop=True))
})

In [30]:
from transformers import (
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
)
MODEL_NAME = "indolem/indobertweet-base-uncased"
tokenizer_s2 = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_s2(batch):
    return tokenizer_s2(batch["processed_text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets_2 = dataset_s2.map(tokenize_s2, batched=True)
tokenized_datasets_2 = tokenized_datasets_2.remove_columns(["processed_text"])
tokenized_datasets_2 = tokenized_datasets_2.rename_column("label", "labels")
tokenized_datasets_2.set_format("torch")

train_dataset_2 = tokenized_datasets_2["train"]
val_dataset_2 = tokenized_datasets_2["validation"]
test_dataset_2 = tokenized_datasets_2["test"]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/3514 [00:00<?, ? examples/s]

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/480 [00:00<?, ? examples/s]

In [36]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, auc, roc_curve, precision_recall_curve
)

def compute_metrics_s2(eval_pred):
    logits, labels = eval_pred
    preds = (logits > 0).astype(int)
    return {
        "f1_micro": f1_score(labels, preds, average="micro"),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
        "recall_micro": recall_score(labels, preds, average="micro", zero_division=0),
        "precision_micro": precision_score(labels, preds, average="micro", zero_division=0),
    }

## Optuna search — Stage 2 (oversampled)

In [ ]:
N_TRIALS = 10
def model_init_s2():
    return AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABEL_COLS),
        problem_type="multi_label_classification"
    )

def objective_s2(trial):
    import shutil, os
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
    epochs = trial.suggest_int("num_train_epochs", 3, 6)
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [8, 16])
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)

    args = TrainingArguments(
        output_dir=f"./optuna_s2/trial_{trial.number}",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=16,
        learning_rate=lr,
        weight_decay=weight_decay,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        save_only_model=True,      # don't persist optimizer state during search (saves disk)
        save_total_limit=1,        # keep only the best checkpoint per trial
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=50,
        disable_tqdm=True,
        report_to=[],
    )

    trainer = Trainer(
        model_init=model_init_s2,
        args=args,
        train_dataset=train_dataset_2,
        eval_dataset=val_dataset_2,
        compute_metrics=compute_metrics_s2,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    try:
        trainer.train()
        metrics = trainer.evaluate()
        return metrics["eval_f1_macro"]
    finally:
        # Free disk between trials: Kaggle's working dir fills up fast, and a
        # half-written checkpoint is what triggers the torch.save iostream error.
        shutil.rmtree(f"./optuna_s2/trial_{trial.number}", ignore_errors=True)

study_s2 = optuna.create_study(direction="maximize", study_name="stage2_oversampling")
study_s2.optimize(objective_s2, n_trials=N_TRIALS)

print("Best Stage 2 params:", study_s2.best_params)
print("Best Stage 2 validation macro F1:", study_s2.best_value)

[I 2026-09-23 02:18:36,453] A new study created in memory with name: stage2_oversampling


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'loss': '1.074', 'grad_norm': '2.585', 'learning_rate': '1.945e-05', 'epoch': '0.2273'}
{'loss': '0.2939', 'grad_norm': '3.782', 'learning_rate': '9.27e-06', 'epoch': '2.727'}
{'loss': '0.2859', 'grad_norm': '4.245', 'learning_rate': '8.344e-06', 'epoch': '2.955'}
{'eval_loss': '0.8305', 'eval_f1_micro': '0.5847', 'eval_f1_macro': '0.5145', 'eval_accuracy': '0.3167', 'eval_recall_micro': '0.6431', 'eval_precision_micro': '0.5359', 'eval_runtime': '1.292', 'eval_samples_per_second': '185.7', 'eval_steps_per_second': '6.19', 'epoch': '3'}


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
[I 2026-09-23 02:58:31,703] Trial 8 finished with value: 0.4801714964996838 and parameters: {'learning_rate': 1.164518073180831e-05, 'num_train_epochs': 6, 'per_device_train_batch_size': 16, 'weight_decay': 0.09601088018815887}. Best is trial 0 with value: 0.5165381168196418.


{'eval_loss': '0.7508', 'eval_f1_micro': '0.5521', 'eval_f1_macro': '0.4802', 'eval_accuracy': '0.3083', 'eval_recall_micro': '0.5608', 'eval_precision_micro': '0.5437', 'eval_runtime': '1.264', 'eval_samples_per_second': '189.9', 'eval_steps_per_second': '6.328', 'epoch': '6'}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

{'loss': '0.6171', 'grad_norm': '3.997', 'learning_rate': '1.929e-05', 'epoch': '0.9091'}


## Train final Stage 2 model with best hyperparameters

In [52]:
study_s2.best_value

0.5182309871144598

In [42]:
best_s2 = study_s2.best_params

args_s2 = TrainingArguments(
    output_dir="./best_s2",
    num_train_epochs=best_s2["num_train_epochs"],
    per_device_train_batch_size=best_s2["per_device_train_batch_size"],
    per_device_eval_batch_size=16,
    learning_rate=best_s2["learning_rate"],
    weight_decay=best_s2["weight_decay"],
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    disable_tqdm=False,
    report_to=[],
)

model_s2 = model_init_s2()
trainer_s2 = Trainer(
    model=model_s2,
    args=args_s2,
    train_dataset=train_dataset_2,
    eval_dataset=val_dataset_2,
    compute_metrics=compute_metrics_s2,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer_s2.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Accuracy,Recall Micro,Precision Micro
1,0.591495,0.718261,0.566406,0.464425,0.316667,0.568627,0.564202
2,0.387164,0.742658,0.539877,0.475158,0.320833,0.517647,0.564103
3,0.241123,0.883051,0.593128,0.512766,0.362500,0.643137,0.550336
4,0.158541,0.904344,0.571970,0.505879,0.333333,0.592157,0.553114
5,0.118384,1.020501,0.580292,0.481286,0.350000,0.623529,0.542662


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1100, training_loss=0.3454629135131836, metrics={'train_runtime': 348.6816, 'train_samples_per_second': 60.468, 'train_steps_per_second': 3.786, 'total_flos': 1155746440773120.0, 'train_loss': 0.3454629135131836, 'epoch': 5.0})

In [43]:
# Per-label threshold tuning on VALIDATION.
pred2_val = trainer_s2.predict(val_dataset_2)
probs_val = 1 / (1 + np.exp(-pred2_val.predictions))
y_val_true = pred2_val.label_ids

best_thresholds_s2 = {}
for i, label_name in enumerate(LABEL_COLS):
    p_curve, r_curve, t_curve = precision_recall_curve(y_val_true[:, i], probs_val[:, i])
    f1_curve = 2 * p_curve * r_curve / (p_curve + r_curve + 1e-12)
    best_idx = np.argmax(f1_curve[:-1])
    best_thresholds_s2[label_name] = float(t_curve[best_idx])
    print(f"{label_name}: best threshold (val)={t_curve[best_idx]:.4f} "
          f"precision={p_curve[best_idx]:.4f} recall={r_curve[best_idx]:.4f} f1={f1_curve[best_idx]:.4f}")

threshold_vector_s2 = np.array([best_thresholds_s2[c] for c in LABEL_COLS])

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


insults: best threshold (val)=0.1717 precision=0.5576 recall=0.8519 f1=0.6740
identity_attack: best threshold (val)=0.2668 precision=0.4835 recall=0.9072 f1=0.6308
threat_incitement_to_violence: best threshold (val)=0.6108 precision=0.3636 recall=0.3636 f1=0.3636
profanity_obscenity: best threshold (val)=0.1780 precision=0.5179 recall=0.8056 f1=0.6304
sexually_explicit: best threshold (val)=0.9730 precision=1.0000 recall=0.3333 f1=0.5000


In [44]:
pred2 = trainer_s2.predict(test_dataset_2)
probs = 1 / (1 + np.exp(-pred2.predictions))
y_true = pred2.label_ids

pred_label2 = (probs > 0.5).astype(int)
pred_label2_tuned = (probs >= threshold_vector_s2).astype(int)

print("Stage 2 classification report @ default threshold (0.5) -- test set:")
print(classification_report(y_true, pred_label2, target_names=LABEL_COLS))

print("\nStage 2 classification report @ per-label tuned thresholds -- test set:")
print(classification_report(y_true, pred_label2_tuned, target_names=LABEL_COLS))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Stage 2 classification report @ default threshold (0.5) -- test set:
                               precision    recall  f1-score   support

                      insults       0.68      0.56      0.62       218
              identity_attack       0.54      0.80      0.65       199
threat_incitement_to_violence       0.28      0.26      0.27        19
          profanity_obscenity       0.58      0.58      0.58        65
            sexually_explicit       0.86      0.55      0.67        11

                    micro avg       0.59      0.65      0.62       512
                    macro avg       0.59      0.55      0.56       512
                 weighted avg       0.60      0.65      0.61       512
                  samples avg       0.56      0.57      0.54       512


Stage 2 classification report @ per-label tuned thresholds -- test set:
                               precision    recall  f1-score   support

                      insults       0.57      0.84      0.68       218
  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

In [47]:
trainer_s2.save_model("./model_stage2_oversampling_threshold_weighted_optuna")
tokenizer_s2.save_pretrained("./model_stage2_oversampling_threshold_weighted_optuna")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./model_stage2_oversampling_threshold_weighted_optuna/tokenizer_config.json',
 './model_stage2_oversampling_threshold_weighted_optuna/tokenizer.json')

In [49]:
from huggingface_hub import HfApi

api = HfApi()

# api.create_repo(
#     repo_id="evelnap/hs-stage2-oversampling_threshold-optuna",
#     repo_type="model",
#     exist_ok=True
# )

api.upload_folder(
    folder_path="./model_stage2_oversampling_threshold_weighted_optuna",
    repo_id="evelnap/hs-stage2-oversampling_threshold-optuna",
    repo_type="model"
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/evelnap/hs-stage2-oversampling_threshold-optuna/commit/7e88af34b4315ed98f53b384d379758464170be5', commit_message='Upload folder using huggingface_hub', commit_description='', oid='7e88af34b4315ed98f53b384d379758464170be5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/evelnap/hs-stage2-oversampling_threshold-optuna', endpoint='https://huggingface.co', repo_type='model', repo_id='evelnap/hs-stage2-oversampling_threshold-optuna'), pr_revision=None, pr_num=None)